In [1]:
import os

In [10]:
def get_source_and_header_files(directory):
    """
    Get all .c, .C, .h, .H file paths in the specified directory and its subdirectories.
    
    Args:
        directory (str): The root directory to search.
    
    Returns:
        list: A list of paths to .c, .C, .h, .H files.
    """
    extensions = {'.c', '.C', '.h', '.H'}
    file_paths = []

    for root, _, files in os.walk(directory):
        for file in files:
            if os.path.splitext(file)[1] in extensions:
                file_paths.append(os.path.join(root, file))

    return file_paths

In [11]:
files = get_source_and_header_files('./repositories/Versa')

In [12]:
files

['./repositories/Versa\\Common.h',
 './repositories/Versa\\COMPAT.C',
 './repositories/Versa\\COMPAT.h',
 './repositories/Versa\\FSHELL.C',
 './repositories/Versa\\FSHELL.H',
 './repositories/Versa\\FV2000.H',
 './repositories/Versa\\RC04EX_AGR.h',
 './repositories/Versa\\RC04EX_apg.c',
 './repositories/Versa\\RC04EX_apg.h',
 './repositories/Versa\\RC04EX_CGR.c',
 './repositories/Versa\\RC04EX_CGR.h',
 './repositories/Versa\\RC04EX_cyc.h',
 './repositories/Versa\\RC04EX_def.c',
 './repositories/Versa\\RC04EX_def.h',
 './repositories/Versa\\RC04EX_fnc.c',
 './repositories/Versa\\RC04EX_fnc.h',
 './repositories/Versa\\RC04EX_FT1.c',
 './repositories/Versa\\RC04EX_FT2.c',
 './repositories/Versa\\RC04EX_iif.h',
 './repositories/Versa\\RC04EX_man.c',
 './repositories/Versa\\RC04EX_man.h',
 './repositories/Versa\\RC04EX_oif.h',
 './repositories/Versa\\RC04EX_pdf.h',
 './repositories/Versa\\RC04EX_pro.h',
 './repositories/Versa\\RC04EX_pro_EUROPA.h',
 './repositories/Versa\\RC04EX_QT_NoSet.c'

In [13]:
import networkx as nx
import json
from networkx.readwrite import json_graph
def json_to_graph(json_format):
    graph_js = json.loads(json_format)
    graph = json_graph.node_link_graph(graph_js)
    return graph

In [14]:
dict_format = json_to_graph('context_database\\Versa.jsonl')

JSONDecodeError: Expecting value: line 1 column 1 (char 0)

In [ ]:
## create graph
from pyvis.network import Network

g = index.get_networkx_graph()
net = Network(notebook=True, cdn_resources="in_line", directed=True)
net.from_nx(g)
net.save_graph("non_filtered_graph.html")

from IPython.display import HTML

HTML(filename="non_filtered_graph.html")

In [7]:
with open('./repositories/devchat/devchat/assistant.py', 'r',encoding='unicode_escape') as f:
    src = f.readlines()

In [9]:
def read_c_file_preserve_strings(file_path):
    lines = []
    with open(file_path, 'r', encoding='unicode_escape', newline='') as f:
        in_string = False
        current_line = []
        line_buffer = []
        
        while True:
            char = f.read(1)
            if not char:  # End of file
                if line_buffer:
                    lines.append(''.join(line_buffer))
                break
            
            # Handle string literal detection
            if char == '"' and (not line_buffer or line_buffer[-1] != '\\'):
                in_string = not in_string
            
            line_buffer.append(char)
            
            # Only split into new lines when we're not inside a string literal
            if char == '\n' and not in_string:
                lines.append(''.join(line_buffer))
                line_buffer = []
                
    return lines

In [10]:
lines = read_c_file_preserve_strings('./repositories/devchat/devchat/assistant.py')

In [25]:
with open('./repositories/versa\FSHELL.C', 'r',encoding='unicode_escape') as f:
    src = f.read()

In [14]:
from tree_sitter import Language, Parser
import tree_sitter_c as tsc
C_LANGUAGE = Language(tsc.language())
parser = Parser(C_LANGUAGE)

# Language.build_library('./my-languages.so', ['./tree-sitter-c'])
# language = Language('./my-languages.so', 'c')
# parser.set_language(language)


In [8]:
import re
def separate_code_and_comment(line):
    """
    Separates a C code line containing a comment into two lines.

    Parameters:
        line (str): The input line of C code.

    Returns:
        tuple: A tuple containing the code part and the comment part as separate strings.
    """
    # Regular expression to match code and comment (// or /* */)
    match = re.match(r'(.*?)(//.*|/\*.*?\*/)', line)
    if match:
        code = match.group(1).rstrip() + "\n"  # Code part with newline added
        comment = match.group(2).lstrip() + "\n"  # Comment part with newline added
        return code, comment
    return line.strip() + "\n", None  # Return the line with newline if no comment found

def process_c_file(lines):
    """
    Process C file through a pipeline of transformations while preserving indentation:
    1. Clean printf statements (remove \n)
    2. Separate code and comments
    3. Convert block comments to single-line comments
    4. Save the processed file
    """



    # Step 1: Clean printf statements while preserving indentation
    printf_cleaned_lines = []
    i = 0
    while i < len(lines):
        current_line = lines[i]
        # Preserve original indentation by not stripping the line
        stripped_line = current_line.rstrip('\n').strip()
        indent = current_line[:len(current_line) - len(current_line.lstrip())]
        
        if 'printf' in stripped_line and not stripped_line.endswith(';'):
            # Handle multi-line printf
            full_printf = stripped_line
            j = i + 1
            while j < len(lines) and ';' not in lines[j]:
                full_printf += ' ' + lines[j].strip()
                j += 1
            if j < len(lines):
                full_printf += ' ' + lines[j].strip()
            
            if 'printf' in full_printf:
                if 'printf("\\n")' in full_printf or 'printf("\\n\\n")' in full_printf:
                    full_printf = full_printf.replace('printf("\\n")', 'printf("")').replace('printf("\\n\\n")', 'printf("")')
                else:
                    parts = full_printf.split('printf')
                    prefix = parts[0]
                    printf_part = 'printf' + parts[1]
                    if '"' in printf_part:
                        start_quote = printf_part.find('"')
                        end_quote = printf_part.rfind('"')
                        if start_quote != -1 and end_quote != -1:
                            string_content = printf_part[start_quote:end_quote+1]
                            string_content = string_content.replace('\\n"', '"').replace('"\\n', '"').replace('\\n', '')
                            printf_part = printf_part[:start_quote] + string_content + printf_part[end_quote+1:]
                    full_printf = prefix + printf_part
            
            printf_cleaned_lines.append(indent + full_printf + '\n')
            i = j + 1
        else:
            if 'printf' in stripped_line:
                if 'printf("\\n");' in stripped_line or 'printf("\\n\\n");' in stripped_line:
                    stripped_line = stripped_line.replace('printf("\\n");', 'printf("");').replace('printf("\\n\\n");', 'printf("");')
                else:
                    parts = stripped_line.split('printf')
                    prefix = parts[0]
                    printf_part = 'printf' + parts[1]
                    if '"' in printf_part:
                        start_quote = printf_part.find('"')
                        end_quote = printf_part.rfind('"')
                        if start_quote != -1 and end_quote != -1:
                            string_content = printf_part[start_quote:end_quote+1]
                            string_content = string_content.replace('\\n"', '"').replace('"\\n', '"').replace('\\n', '')
                            printf_part = printf_part[:start_quote] + string_content + printf_part[end_quote+1:]
                    stripped_line = prefix + printf_part
                printf_cleaned_lines.append(indent + stripped_line + '\n')
            else:
                printf_cleaned_lines.append(current_line)
            i += 1

    # Step 2: Modify separate_comment_c_code_lines to preserve indentation
    def separate_comment_c_code_lines(lines):
        processed_lines = []
        for line in lines:
            original_indent = line[:len(line) - len(line.lstrip())]
            code, comment = separate_code_and_comment(line)
            if code:
                processed_lines.append(original_indent + code.lstrip())
            if comment:
                processed_lines.append(original_indent + comment.lstrip())
        return processed_lines

    # Step 3: Modify convert_docs_to_comments to preserve indentation
    def convert_docs_to_comments_with_indent(lines):
        merged_lines = []
        is_in_block_comment = False
        current_comment = ''
        original_indent = ''
        
        single_line_comment = '//'
        block_comment_start = '/*'
        block_comment_end = '*/'
        
        i = 0
        while i < len(lines):
            line = lines[i]
            original_indent = line[:len(line) - len(line.lstrip())]
            
            if block_comment_start in line and block_comment_end in line:
                parts = line.split('/*')
                code_part = parts[0]
                comment_part = parts[1].split('*/')[0].strip()
                rest_of_line = line[line.index('*/') + 2:]
                
                new_line = ''
                if code_part.strip():
                    new_line += original_indent + code_part.lstrip()
                
                if comment_part.strip():
                    if code_part.strip():
                        if len(code_part.rstrip()) < 40:
                            new_line = new_line.rstrip() + ' ' * (40 - len(new_line.rstrip()))
                        else:
                            new_line = new_line.rstrip() + ' ' * 4
                    else:
                        new_line += original_indent
                    new_line += f"{single_line_comment} {comment_part}"
                
                if rest_of_line.strip():
                    new_line += rest_of_line
                
                if line.endswith('\n'):
                    new_line += '\n'
                
                merged_lines.append(new_line)
                
            elif block_comment_start in line and not is_in_block_comment:
                is_in_block_comment = True
                code_part = line[:line.index(block_comment_start)]
                if code_part.strip():
                    merged_lines.append(original_indent + code_part.lstrip())
                start_idx = line.index(block_comment_start)
                current_comment = line[start_idx + len(block_comment_start):].strip()
                
            elif block_comment_end in line and is_in_block_comment:
                is_in_block_comment = False
                end_idx = line.index(block_comment_end)
                current_comment += ' ' + line[:end_idx].strip()
                
                comment_line = f"{original_indent}{single_line_comment} {current_comment.strip()}"
                if line.endswith('\n'):
                    comment_line += '\n'
                merged_lines.append(comment_line)
                
                code_after = line[end_idx + 2:]
                if code_after.strip():
                    merged_lines.append(original_indent + code_after.lstrip())
                
                current_comment = ''
                
            elif is_in_block_comment:
                current_comment += ' ' + line.strip().lstrip('* ')
                
            else:
                merged_lines.append(line)
            
            i += 1
        
        return merged_lines

    # Apply the pipeline while preserving indentation
    separated_lines = separate_comment_c_code_lines(printf_cleaned_lines)
    final_lines = convert_docs_to_comments_with_indent(separated_lines)

    return final_lines

In [9]:
with open('./data/versa\RC04EX_WT.c', 'r',encoding='unicode_escape') as f:
    src_lines = f.readlines()

In [10]:
len(src_lines)

1200

In [11]:
src_lines = process_c_file(src_lines)

In [12]:
src_lines = "".join(src_lines).encode('ascii', errors='ignore').decode('ascii')
src_lines = src_lines.splitlines(keepends=True)
if len(src_lines) != 0:
    src_lines[-1] = src_lines[-1].rstrip().strip('(').strip('[').strip(',')
comment_lines = []
comment_prefix = "//"
for i in range(0, len(src_lines)):
        line = src_lines[i]
        if line.lstrip().startswith(comment_prefix):
            src_lines[i] = '\n'
            comment_lines.append(i)

In [15]:
def read_callable_point(byte_offset, point):
    row, column = point
    if row >= len(src_lines) or column >= len(src_lines[row]):
        return None
    return src_lines[row][column:].encode("utf8")


tree = parser.parse(read_callable_point, encoding="utf8")

In [16]:
tree.root_node.children

[<Node type=preproc_include, start_point=(1, 0), end_point=(2, 0)>,
 <Node type=preproc_include, start_point=(2, 0), end_point=(3, 0)>,
 <Node type=preproc_include, start_point=(3, 0), end_point=(4, 0)>,
 <Node type=preproc_include, start_point=(4, 0), end_point=(5, 0)>,
 <Node type=preproc_include, start_point=(5, 0), end_point=(6, 0)>,
 <Node type=preproc_include, start_point=(6, 0), end_point=(7, 0)>,
 <Node type=preproc_include, start_point=(7, 0), end_point=(8, 0)>,
 <Node type=preproc_include, start_point=(9, 0), end_point=(10, 0)>,
 <Node type=preproc_include, start_point=(10, 0), end_point=(11, 0)>,
 <Node type=preproc_include, start_point=(11, 0), end_point=(12, 0)>,
 <Node type=preproc_include, start_point=(13, 0), end_point=(14, 0)>,
 <Node type=preproc_include, start_point=(15, 0), end_point=(16, 0)>,
 <Node type=preproc_include, start_point=(16, 0), end_point=(17, 0)>,
 <Node type=preproc_include, start_point=(17, 0), end_point=(18, 0)>,
 <Node type=preproc_include, start_

In [17]:
src_lines[122]

'int WaferTestProgram( int prob_exec ){\n'

In [216]:
def print_tree(node, indent=0):
    """Recursively print the tree structure."""
    # Create indent spacing
    prefix = "  " * indent

    # Display the node with type and position
    node_info = f"{node.type} [{node.start_point}] - [{node.end_point}]"
    print(prefix + node_info)

    # Iterate over children and include field names if present
    for i, child in enumerate(node.children):
        field_name = node.field_name_for_child(i)  # Use the index `i` here
        if field_name:
            print(f"{prefix}  {field_name}: ", end="")
        print_tree(child, indent + (2 if field_name else 1))


In [116]:
print_tree(tree.root_node)

translation_unit [Point(row=2, column=0)] - [Point(row=793, column=0)]
  declaration [Point(row=2, column=0)] - [Point(row=2, column=21)]
    type:       primitive_type [Point(row=2, column=0)] - [Point(row=2, column=3)]
    declarator:       identifier [Point(row=2, column=4)] - [Point(row=2, column=20)]
    ; [Point(row=2, column=20)] - [Point(row=2, column=21)]
  declaration [Point(row=3, column=0)] - [Point(row=3, column=39)]
    type:       primitive_type [Point(row=3, column=0)] - [Point(row=3, column=4)]
    declarator:       init_declarator [Point(row=3, column=5)] - [Point(row=3, column=38)]
        declarator:           array_declarator [Point(row=3, column=5)] - [Point(row=3, column=20)]
            declarator:               identifier [Point(row=3, column=5)] - [Point(row=3, column=18)]
            [ [Point(row=3, column=18)] - [Point(row=3, column=19)]
            ] [Point(row=3, column=19)] - [Point(row=3, column=20)]
        = [Point(row=3, column=21)] - [Point(row=3, co

In [17]:
src ='''
void StringInput( char *str )
{
  int n, i;
  char  buf[1024];

  fflush( stdout );
  buf[0] = NUL;
  for( i = 0; NULL == fgets(buf, 1024, stdin) && i < 5; i++ );
  n = strlen( buf ) - 1;
  if( n >= 0 && '\n' == buf[n] )  buf[n] = NUL; /* cut last LF */
  strcpy( str, buf );
}
'''

In [18]:
tree_test = parser.parse(bytes(src,encoding='utf-8'))
tree_test.root_node.children

[<Node type=function_definition, start_point=(1, 0), end_point=(13, 1)>]

In [185]:
clean_printf_newlines('./repositories/versa\FSHELL.C')

Successfully cleaned printf newlines in ./repositories/versa\FSHELL.C.


In [5]:
import os
import json
import tiktoken
from typing import List, Dict, Any, Tuple
from dataclasses import dataclass
import time

@dataclass
class CONSTANTS:
    max_search_top_k: int = 10
    graph_database_save_dir: str = "./context_database"

class CodexTokenizer:
    def __init__(self):
        self.tokenizer = tiktoken.encoding_for_model("gpt-3.5")
    
    def tokenize(self, text: str) -> List[int]:
        return self.tokenizer.encode(text, allowed_special={'<|endoftext|>'})
    
    def decode(self, token_ids: List[int]) -> str:
        return self.tokenizer.decode(token_ids)

class SimilarityScore:
    @staticmethod
    def text_jaccard_similarity(list1: List[int], list2: List[int]) -> float:
        set1 = set(list1)
        set2 = set(list2)
        intersection = len(set1.intersection(set2))
        union = len(set1.union(set2))
        return float(intersection) / union if union > 0 else 0.0

class CodeSearchInference:
    def __init__(self, database_dir: str = CONSTANTS.graph_database_save_dir):
        self.database_dir = database_dir
        self.tokenizer = CodexTokenizer()
        self.constants = CONSTANTS()
    
    def load_database(self, repo_name: str) -> List[Dict[str, Any]]:
        """Load the code database for a specific repository"""
        database_path = os.path.join(self.database_dir, f"{repo_name}.jsonl")
        if not os.path.exists(database_path):
            raise FileNotFoundError(f"Database file not found: {database_path}")
            
        database = []
        with open(database_path, 'r', encoding='utf-8') as f:
            for line in f:
                database.append(json.loads(line.strip()))
        return database

    def search(self, query: str, repo_name: str) -> Tuple[List[Dict[str, Any]], float]:
        """
        Search for code snippets similar to the natural language query
        
        Args:
            query: Natural language query
            repo_name: Name of the repository to search in
            
        Returns:
            Tuple of (list of results, search time)
        """
        start_time = time.time()
        
        # Load and tokenize query
        query_tokens = self.tokenizer.tokenize(query)
        
        # Load database
        try:
            database = self.load_database(repo_name)
        except FileNotFoundError as e:
            print(f"Error: {e}")
            return [], 0.0
        
        # Calculate similarity scores
        search_results = []
        for case in database:
            sim_score = SimilarityScore.text_jaccard_similarity(
                query_tokens,
                case['key_forward_encoding']
            )
            if sim_score > 0:  # Only include non-zero similarities
                result = {
                    'val': case['val'],
                    'statement': case['statement'],
                    'key_forward_context': case['key_forward_context'],
                    'key_forward_graph': case['key_forward_graph'],
                    'key_forward_encoding': case['key_forward_encoding'],
                    'fpath_tuple': case['fpath_tuple'],
                    'similarity_score': sim_score
                }
                search_results.append(result)
        
        # Sort by similarity score and get top-k results
        search_results.sort(key=lambda x: x['similarity_score'], reverse=True)
        top_k_results = search_results[:self.constants.max_search_top_k]
        
        search_time = time.time() - start_time
        return top_k_results, search_time

def format_results(results: List[Dict[str, Any]], search_time: float) -> str:
    """Format search results for display"""
    output = f"Search completed in {search_time:.4f} seconds\n"
    output += f"Found {len(results)} relevant code snippets\n\n"
    
    for i, result in enumerate(results, 1):
        output += f"Result {i} (Similarity: {result['similarity_score']:.4f})\n"
        output += f"File: {'/'.join(result['fpath_tuple'])}\n"
        output += "Code Context:\n"
        output += result['val'] + "\n"
        output += "-" * 80 + "\n"
    
    return output



In [6]:
searcher = CodeSearchInference()
query = "int comp_write_bm( int address, int data ){"
repo_name = "versa"  # Replace with your repository name

results, search_time = searcher.search(query, repo_name)
print(format_results(results, search_time))

Search completed in 0.6973 seconds
Found 10 relevant code snippets

Result 1 (Similarity: 0.8182)
File: COMPAT.C
Code Context:

void comp_copy_ecr_to_buf( int start_addr, int end_addr ){
  int data_size;
#if defined(V5KONLY)
  data_size = end_addr - start_addr + 1;
  copy_ecr_to_buf (start_addr, data_size);
#else
#endif
}


int comp_write_bm( int address, int data ){
  int status;
  status = 0;
#ifdef V4K5KTESTER
  status = write_bmecr( bm1Memory, address, data );
#else
  write_bm( address, data );
#endif
  return( status );
}

--------------------------------------------------------------------------------
Result 2 (Similarity: 0.5000)
File: COMPAT.C
Code Context:
  int status;
  status = 0;
#ifdef V4K5KTESTER
  status = write_bmecr( bm1Memory, address, data );
#else
  write_bm( address, data );
#endif
  return( status );
}


int comp_read_bm( int address ){
  int status;
  status = 0;
#ifdef V4K5KTESTER
  status = read_bmecr( bm1Memory, address );
#else
  read_bm( address );
#endif
 

In [14]:
import os
import json
import numpy as np
import tiktoken
from typing import List, Dict, Any, Tuple, Literal
from dataclasses import dataclass
import time
from openai import OpenAI
from enum import Enum

class SearchMode(Enum):
    TEXT = "text"
    EMBEDDING = "embedding"
    COMBINED = "combined"

@dataclass
class CONSTANTS:
    max_search_top_k: int = 10
    graph_database_save_dir: str = "./context_database"
    # Weight for combining scores (embedding_weight)
    # Higher value gives more importance to embedding similarity
    embedding_weight: float = 0.7

class CodexTokenizer:
    def __init__(self):
        self.tokenizer = tiktoken.encoding_for_model("gpt-3.5")
    
    def tokenize(self, text: str) -> List[int]:
        return self.tokenizer.encode(text, allowed_special={'<|endoftext|>'})
    
    def decode(self, token_ids: List[int]) -> str:
        return self.tokenizer.decode(token_ids)

class DatabricksEmbedding:
    def __init__(self, databricks_token=None, base_url=None):
        self.token = databricks_token or os.environ.get('DATABRICKS_TOKEN')
        self.base_url = base_url or "https://adb-379144824042062.2.azuredatabricks.net/serving-endpoints"
        
        self.client = OpenAI(
            api_key=self.token,
            base_url=self.base_url
        )
        
    def get_embedding(self, text: str, model: str = "databricks-bge-large-en") -> List[float]:
        response = self.client.embeddings.create(
            input=text,
            model=model
        )
        return response.data[0].embedding

class SimilarityMetrics:
    @staticmethod
    def text_jaccard_similarity(list1: List[int], list2: List[int]) -> float:
        set1 = set(list1)
        set2 = set(list2)
        intersection = len(set1.intersection(set2))
        union = len(set1.union(set2))
        return float(intersection) / union if union > 0 else 0.0

    @staticmethod
    def cosine_similarity(vec1: List[float], vec2: List[float]) -> float:
        vec1 = np.array(vec1)
        vec2 = np.array(vec2)
        
        if np.all(vec1 == 0) or np.all(vec2 == 0):
            return 0.0
            
        cosine_sim = np.dot(vec1, vec2) / (np.linalg.norm(vec1) * np.linalg.norm(vec2))
        return float(cosine_sim)

class CodeSearchInference:
    def __init__(self, 
                 database_dir: str = CONSTANTS.graph_database_save_dir,
                 databricks_token: str = None,
                 databricks_url: str = None):
        self.database_dir = database_dir
        self.constants = CONSTANTS()
        self.tokenizer = CodexTokenizer()
        self.embedding_model = DatabricksEmbedding(
            databricks_token=databricks_token,
            base_url=databricks_url
        )
    
    def load_database(self, repo_name: str) -> List[Dict[str, Any]]:
        """Load the code database for a specific repository"""
        database_path = os.path.join(self.database_dir, f"{repo_name}.jsonl")
        if not os.path.exists(database_path):
            raise FileNotFoundError(f"Database file not found: {database_path}")
            
        database = []
        with open(database_path, 'r', encoding='utf-8') as f:
            for line in f:
                database.append(json.loads(line.strip()))
        return database

    def _get_text_similarity_results(self, query: str, database: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
        """Get results using text-based Jaccard similarity"""
        query_tokens = self.tokenizer.tokenize(query)
        results = []
        
        for case in database:
            sim_score = SimilarityMetrics.text_jaccard_similarity(
                query_tokens,
                case['key_forward_encoding']
            )
            if sim_score > 0:
                result = {**case, 'text_similarity': sim_score}
                results.append(result)
                
        return results

    def _get_embedding_similarity_results(self, query: str, database: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
        """Get results using embedding-based cosine similarity"""
        query_embedding = self.embedding_model.get_embedding(query)
        results = []
        
        for case in database:
            sim_score = SimilarityMetrics.cosine_similarity(
                query_embedding,
                case['key_forward_embedding']
            )
            if sim_score > 0:
                result = {**case, 'embedding_similarity': sim_score}
                results.append(result)
                
        return results

    def search(self, 
               query: str, 
               repo_name: str, 
               mode: SearchMode = SearchMode.COMBINED) -> Tuple[List[Dict[str, Any]], Dict[str, float]]:
        """
        Search for code snippets using specified search mode
        
        Args:
            query: Natural language query
            repo_name: Name of the repository to search in
            mode: Search mode (text, embedding, or combined)
            
        Returns:
            Tuple of (list of results, timing information)
        """
        timing = {}
        start_time = time.time()
        
        try:
            database = self.load_database(repo_name)
        except FileNotFoundError as e:
            print(f"Error: {e}")
            return [], {'total_time': 0.0}

        results = []
        if mode in [SearchMode.TEXT, SearchMode.COMBINED]:
            text_start = time.time()
            text_results = self._get_text_similarity_results(query, database)
            timing['text_search_time'] = time.time() - text_start
            
            if mode == SearchMode.TEXT:
                results = [{
                    'val': r['val'],
                    'statement': r['statement'],
                    'key_forward_context': r['key_forward_context'],
                    'key_forward_graph': r['key_forward_graph'],
                    'fpath_tuple': r['fpath_tuple'],
                    'similarity_score': r['text_similarity']
                } for r in text_results]

        if mode in [SearchMode.EMBEDDING, SearchMode.COMBINED]:
            embedding_start = time.time()
            embedding_results = self._get_embedding_similarity_results(query, database)
            timing['embedding_search_time'] = time.time() - embedding_start
            
            if mode == SearchMode.EMBEDDING:
                results = [{
                    'val': r['val'],
                    'statement': r['statement'],
                    'key_forward_context': r['key_forward_context'],
                    'key_forward_graph': r['key_forward_graph'],
                    'fpath_tuple': r['fpath_tuple'],
                    'similarity_score': r['embedding_similarity']
                } for r in embedding_results]

        if mode == SearchMode.COMBINED:
            # Create a map of results by their content for combining scores
            combined_results = {}
            
            # Process text results
            for r in text_results:
                key = (r['val'], r['statement'])  # Use content as key
                combined_results[key] = {
                    'val': r['val'],
                    'statement': r['statement'],
                    'key_forward_context': r['key_forward_context'],
                    'key_forward_graph': r['key_forward_graph'],
                    'fpath_tuple': r['fpath_tuple'],
                    'text_similarity': r['text_similarity'],
                    'embedding_similarity': 0.0
                }

            # Process embedding results
            for r in embedding_results:
                key = (r['val'], r['statement'])
                if key in combined_results:
                    combined_results[key]['embedding_similarity'] = r['embedding_similarity']
                else:
                    combined_results[key] = {
                        'val': r['val'],
                        'statement': r['statement'],
                        'key_forward_context': r['key_forward_context'],
                        'key_forward_graph': r['key_forward_graph'],
                        'fpath_tuple': r['fpath_tuple'],
                        'text_similarity': 0.0,
                        'embedding_similarity': r['embedding_similarity']
                    }

            # Combine scores using weighted average
            w = self.constants.embedding_weight
            results = [{
                **r,
                'similarity_score': (w * r['embedding_similarity'] + (1-w) * r['text_similarity']),
                'text_similarity': r['text_similarity'],
                'embedding_similarity': r['embedding_similarity']
            } for r in combined_results.values()]

        # Sort by similarity score and get top-k results
        results.sort(key=lambda x: x['similarity_score'], reverse=True)
        results = results[:self.constants.max_search_top_k]
        
        timing['total_time'] = time.time() - start_time
        return results, timing

def format_results(results: List[Dict[str, Any]], timing: Dict[str, float], mode: SearchMode) -> str:
    """Format search results for display"""
    output = f"Search completed in {timing['total_time']:.4f} seconds\n"
    if 'text_search_time' in timing:
        output += f"Text search time: {timing['text_search_time']:.4f} seconds\n"
    if 'embedding_search_time' in timing:
        output += f"Embedding search time: {timing['embedding_search_time']:.4f} seconds\n"
    output += f"Found {len(results)} relevant code snippets\n\n"
    
    for i, result in enumerate(results, 1):
        output += f"Result {i} (Overall Similarity: {result['similarity_score']:.4f})\n"
        if mode == SearchMode.COMBINED:
            output += f"Text Similarity: {result['text_similarity']:.4f}\n"
            output += f"Embedding Similarity: {result['embedding_similarity']:.4f}\n"
        output += f"File: {'/'.join(result['fpath_tuple'])}\n"
        output += "Code Context:\n"
        output += result['val'] + "\n"
        output += "-" * 80 + "\n"
    
    return output


In [ ]:
# Example usage
databricks_token = os.environ.get("DATABRICKS_TOKEN", "")

searcher = CodeSearchInference(databricks_token=databricks_token)


# Try all search modes
# for mode in SearchMode:
#     print(f"\nTrying {mode.value} search mode:")
#     results, timing = searcher.search(query, repo_name, mode=mode)
#     print(format_results(results, timing, mode))

In [25]:
query = """ 
Function to PowerDown device, set vih1 to 0
"""
repo_name = "versa"

results, timing = searcher.search(query, repo_name, mode=SearchMode.EMBEDDING)

In [26]:
print(format_results(results, timing, SearchMode.EMBEDDING))

Search completed in 9.9848 seconds
Embedding search time: 2.7797 seconds
Found 10 relevant code snippets

Result 1 (Overall Similarity: 0.7640)
File: RC04EX_apg.c
Code Context:
  return;
}

 // *********************************************************************** DeviceLevelsPowerDown Function ***********************************************************************
int DeviceLevelsPowerDown(void)
{

  set_vih1(0);
  set_vil2(0);
  set_vhh(0);
  comp_set_v4(0);
  comp_set_v3(0);
  comp_set_v2(0);
  comp_set_v1(0);

  set_vlimit(0);
  comp_set_ilimit(1);
  reconnect_pin(v4_pin);

  set_invert_mask(no_pin);
#ifdef V5KONLY

--------------------------------------------------------------------------------
Result 2 (Overall Similarity: 0.7614)
File: RC04EX_apg.c
Code Context:

  return;
}

 // *********************************************************************** DeviceLevelsPowerDown Function ***********************************************************************
int DeviceLevelsPowerDown(

In [34]:
import os
import argparse
import json
from tree_sitter import Language, Parser
import tree_sitter_c as tsc
def setup_parser():
    # You need to build the tree-sitter-c library first and place it in the specified path
    C_LANGUAGE = Language(tsc.language())
    parser = Parser(C_LANGUAGE)
    return parser

def extract_code_elements(file_content, parser):
    tree = parser.parse(bytes(file_content, "utf8"))
    root_node = tree.root_node
    
    functions = []
    includes = []
    preprocessors = []
    structs = []
    enums = []
    
    def extract_preprocessor_block(node):
        """Extract entire preprocessor block including all nested content"""
        start_byte = node.start_byte
        # Find the end of the preprocessor block
        current = node
        while current.next_sibling and current.next_sibling.type.startswith('preproc_'):
            current = current.next_sibling
        end_byte = current.end_byte
        return file_content[start_byte:end_byte]
    
    def process_node(node):
        if node.type == "function_definition":
            functions.append({
                'original_code': file_content[node.start_byte:node.end_byte],
                'type': 'function'
            })
        elif node.type == "preproc_include":
            includes.append({
                'original_code': file_content[node.start_byte:node.end_byte],
                'type': 'include'
            })
        elif node.type in ["preproc_if", "preproc_ifdef", "preproc_ifndef"]:
            block_content = extract_preprocessor_block(node)
            preprocessors.append({
                'original_code': block_content,
                'type': node.type,
                'condition': file_content[node.start_byte:node.children[1].end_byte] if len(node.children) > 1 else None
            })
        elif node.type == "struct_specifier":
            # Find the struct name if it exists
            name_node = next((child for child in node.children if child.type == "type_identifier"), None)
            struct_name = file_content[name_node.start_byte:name_node.end_byte] if name_node else "anonymous"
            structs.append({
                'original_code': file_content[node.start_byte:node.end_byte],
                'type': 'struct',
                'name': struct_name
            })
        elif node.type == "enum_specifier":
            # Find the enum name if it exists
            name_node = next((child for child in node.children if child.type == "type_identifier"), None)
            enum_name = file_content[name_node.start_byte:name_node.end_byte] if name_node else "anonymous"
            enums.append({
                'original_code': file_content[node.start_byte:node.end_byte],
                'type': 'enum',
                'name': enum_name
            })
            
        # Recursively process children
        for child in node.children:
            process_node(child)
    
    # Start processing from the root
    process_node(root_node)
    
    return {
        'functions': functions,
        'includes': includes,
        'preprocessors': preprocessors,
        'structs': structs,
        'enums': enums
    }

def chunk_function(c_code, parser):
    # Parse the function code
    tree = parser.parse(bytes(c_code, "utf8"))
    root_node = tree.root_node
    chunks = []
    current_chunk = []
    
    def process_node(node, depth=0):
        nonlocal chunks, current_chunk
        
        # Define which node types should start a new chunk
        chunk_starters = {
            "compound_statement",  # Block statements
            "if_statement",
            "for_statement",
            "while_statement",
            "do_statement",
            "switch_statement",
            "case_statement"
        }
        
        if node.type in chunk_starters and depth > 0:  # Don't chunk the function's main compound statement
            if current_chunk:
                chunks.append("\n".join(current_chunk))
                current_chunk = []
            
            # Add this control structure and its content as a new chunk
            current_chunk.append(c_code[node.start_byte:node.end_byte])
            chunks.append("\n".join(current_chunk))
            current_chunk = []
        else:
            # For leaf nodes that aren't part of control structures
            if len(node.children) == 0 and node.type != "comment":
                current_chunk.append(c_code[node.start_byte:node.end_byte])
            
            # Recursively process children
            for child in node.children:
                process_node(child, depth + 1)
    
    # Start processing from the root
    process_node(root_node)
    
    # Add any remaining chunk
    if current_chunk:
        chunks.append("\n".join(current_chunk))
    
    return [chunk for chunk in chunks if chunk.strip()]

def merge_chunks(chunks, max_lines=15):
    merged_chunks = []
    current_chunk = []
    
    for chunk in chunks:
        chunk_lines = chunk.split('\n')
        
        # If adding this chunk will not exceed the line limit, add it
        if len(current_chunk) + len(chunk_lines) <= max_lines:
            current_chunk.extend(chunk_lines)
        else:
            # If adding exceeds limit, finalize current chunk and start a new one
            if current_chunk:
                merged_chunks.append("\n".join(current_chunk))
            current_chunk = chunk_lines
    
    # Add any remaining lines in the final chunk
    if current_chunk:
        merged_chunks.append("\n".join(current_chunk))
    
    return merged_chunks

def extract_and_chunk_code(folder_path, output_path):
    parser = setup_parser()
    json_code = []
    
    # List all .c files in the folder with full paths
    c_files = [os.path.join(folder_path, f) for f in os.listdir(folder_path) if f.endswith('.c')]
    
    # Iterate over each .c file
    for file_path in c_files:
        file_name = os.path.basename(file_path)
        
        # Read the file content
        with open(file_path, 'r', encoding='latin-1') as file:
            file_content = file.read()
        
        # Extract code elements using tree-sitter
        elements = extract_code_elements(file_content, parser)
        
        # Process each function and chunk it
        for func in elements['functions']:
            chunks = chunk_function(func['original_code'], parser)
            func['chunks'] = merge_chunks(chunks, max_lines=25)
        
        # Append extracted code elements to the json_code list
        for code_type, code_list in elements.items():
            for code in code_list:
                base_entry = {
                    'original_code': code.get('original_code') if isinstance(code, dict) else code,
                    'code_type': code_type,
                    'original_file': file_name
                }
                
                # Add additional metadata if available
                if isinstance(code, dict):
                    if code.get('type'):
                        base_entry['element_type'] = code['type']
                    if code.get('name'):
                        base_entry['name'] = code['name']
                    if code.get('condition'):
                        base_entry['condition'] = code['condition']
                    
                    # Handle chunks for functions
                    if code_type == 'functions' and code.get('chunks') and len(code['chunks']) >= 2:
                        base_entry['chunks'] = code['chunks']
                    else:
                        base_entry['chunks'] = None
                
                json_code.append(base_entry)
    
    # Write the JSON output to the specified file
    with open(output_path, 'w', encoding='utf-8') as file:
        json.dump(json_code, file, ensure_ascii=False, indent=4)
